In [68]:
import pandas as pd
import json
import random

In [69]:
new_triples_dmd = pd.read_csv('../new_triples_dmd_node_exist.csv')
print(new_triples_dmd.shape[0])
new_triples_dmd.head()

945


,relation,x_type,y_type,x_id,y_id
0,disease_phenotype_positive,disease,effect/phenotype,kg4rd:10679,kg4rd:969
1,disease_phenotype_positive,disease,effect/phenotype,kg4rd:10399,kg4rd:8244
2,disease_protein,disease,gene/protein,kg4rd:10679,kg4rd:8864
3,disease_protein,disease,gene/protein,kg4rd:10311,kg4rd:2006
4,disease_phenotype_positive,disease,effect/phenotype,kg4rd:10679,kg4rd:1635


In [70]:
nodes = pd.read_csv('../../../kg/nodes.csv')
print(nodes.shape[0])
nodes.head()

151321


,node_index,node_id,node_type,node_name,node_source
0,0,kg4rd:381,gene/protein,ARF5,NCBI
1,1,kg4rd:4074,gene/protein,M6PR,NCBI
2,2,kg4rd:2288,gene/protein,FKBP4,NCBI
3,3,kg4rd:56603,gene/protein,CYP26B1,NCBI
4,4,kg4rd:55471,gene/protein,NDUFAF7,NCBI


In [71]:
nodes_d = nodes.drop_duplicates(['node_id', 'node_type'], keep='first')
nodes_d.shape[0]

151112

In [73]:
df = pd.merge(new_triples_dmd, nodes_d, left_on=['x_id', 'x_type'], right_on=['node_id', 'node_type'], how='left').rename(columns={'node_index': 'x_index'}).get(
    ['relation', 'x_index', 'y_id', 'y_type']
).astype({'x_index': int}).astype({'x_index': str})

print(df.shape[0])

df = pd.merge(df, nodes_d, left_on=['y_id', 'y_type'], right_on=['node_id', 'node_type'], how='left').rename(columns={'node_index': 'y_index'}).get(
    ['relation', 'x_index', 'y_index']
).astype({'y_index': int}).astype({'y_index': str})

print(df.shape[0])
df.head()

945
945


,relation,x_index,y_index
0,disease_phenotype_positive,27017,29806
1,disease_phenotype_positive,48206,32600
2,disease_protein,27017,1721
3,disease_protein,27018,8674
4,disease_phenotype_positive,27017,30972


In [80]:
edges = pd.read_csv('../../../kg/edges.csv')
print(edges.shape[0])
edges.head()

14641692


,relation,display_relation,x_index,y_index
0,protein_protein,ppi,0,1898
1,protein_protein,ppi,0,769
2,protein_protein,ppi,0,15031
3,protein_protein,ppi,0,2385
4,protein_protein,ppi,0,4981


In [83]:
edges = pd.concat([
    edges,
    df
], ignore_index=True)
print(edges.shape[0])
edges.tail()

14642637


,relation,display_relation,x_index,y_index
14642632,disease_phenotype_positive,NaN,27017,35978
14642633,disease_protein,NaN,27017,8084
14642634,disease_protein,NaN,27017,9733
14642635,disease_disease,NaN,27018,40239
14642636,disease_phenotype_positive,NaN,27017,35139


In [85]:
with open('./data/entity2id.txt', 'w') as f:
    f.write(f'{len(nodes)}\n')
    for _, row in nodes.iterrows():
        f.write(f'{row["node_name"]}:{row["node_type"]}\t{row["node_index"]}\n')

In [86]:
relations = edges.drop_duplicates(subset=['relation'])['relation'].to_list()
relation2id = {relation: i for i, relation in enumerate(relations)}

with open('./data/relation2id.txt', 'w') as f:
    f.write(f'{len(relations)}\n')
    for k, v in relation2id.items():
        f.write(f'{k}\t{v}\n')

In [87]:
edges['rela_id'] = edges['relation'].apply(lambda x: relation2id[x])
edges = edges[['x_index', 'y_index', 'rela_id']]
data = list(edges.itertuples(index=False, name=None))
random.shuffle(data)
data[:10]

[(5506, 95037, 28),
 (10721, 13244, 0),
 (1917, 105137, 28),
 (22948, 19567, 5),
 (95138, 10187, 28),
 (20165, 21361, 5),
 (94957, 6239, 28),
 (22051, 18351, 5),
 (44246, 906, 10),
 (16600, 83771, 17)]

In [92]:
for d in data:
    if int(d[0]) >= 151321 or int(d[1]) >= 151321:
        print(d)

In [90]:
train_size = int(len(data) * 0.9)
valid_size = int(len(data) * 0.05)
test_size = len(data) - train_size - valid_size

print('train size: ', train_size)
print('valid size: ', valid_size)
print('test size: ', test_size)

train_data = data[:train_size]
valid_data = data[train_size:train_size+valid_size]
test_data = data[train_size+valid_size:]

train size:  13178373
valid size:  732131
test size:  732133


In [91]:
with open('./data/train2id.txt', 'w') as f:
    f.write(f'{len(train_data)}\n')
    for x, y, r in train_data:
        f.write(f'{x} {y} {r}\n')

with open('./data/valid2id.txt', 'w') as f:
    f.write(f'{len(valid_data)}\n')
    for x, y, r in valid_data:
        f.write(f'{x} {y} {r}\n')

with open('./data/test2id.txt', 'w') as f:
    f.write(f'{len(test_data)}\n')
    for x, y, r in test_data:
        f.write(f'{x} {y} {r}\n')